# Run Polaris on 25 FOVs in LM39 cell dataset
- 0.85 spot detection probability
- GS registration method
- *No* butter2D filter

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras.utils import get_file

from deepcell.utils.plot_utils import create_rgb_image
from deepcell.datasets import SpotNetExampleData
from deepcell_spots.applications import Polaris
from deepcell_spots.utils.results_utils import mask_spots

import pandas as pd
from typing import Union, List, Dict, Tuple
import re
from skimage.transform import AffineTransform
import csv

# ---- Main folder = the repository folder, which holds Data/ (layout described in Data/README.md). ----
# Run this from the folder the file is in, or set the environment variable GUIDESTAR_MAIN_DIR to the repository path.
main_dir = os.environ.get('GUIDESTAR_MAIN_DIR', '../..')


os.environ["DEEPCELL_ACCESS_TOKEN"] = "<your DeepCell access token>" # Set Deepcell access token

In [2]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  # Restrict TensorFlow to only allocate 1GB of memory on the first GPU
  try:
    tf.config.set_logical_device_configuration(
        gpus[0],
        [tf.config.LogicalDeviceConfiguration(memory_limit=5000)])
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Virtual devices must be set before GPUs have been initialized
    print(e)

1 Physical GPUs, 1 Logical GPUs


Load functions to read Dax files

In [ ]:
import sys
sys.path.append('../..')  # repository folder
from Guidestar_colocalisation.readClasses import DaxRead, readDoryImg

In [6]:
# load dax file [In this case, test 1 FOV test: fov 11]
data_path = os.path.join(main_dir, 'Data/100uM/')
hyb_list = [9,8,7,6,5,4,2,3,9,8,7,6,5,4,3,2]
type_list = ['Cy7']*8 +['Cy5']*8
channel_hyb = [f"{t}_{h:02d}" for t, h in zip(type_list, hyb_list)]
nfov = 25
rounds = 8
channels = 2

df_barcodes = pd.read_csv(os.path.join(main_dir, 'Data/100uM/codebook/codebook_data.tsv'),
                          header=None,
                          sep="\t")

#modify the barcode to fit sample format
last_col = df_barcodes.columns[-1]
df_barcodes = df_barcodes[[last_col] + list(df_barcodes.columns[:-1])]
df_barcodes.columns = ['Gene'] + channel_hyb
df_barcodes

,Gene,Cy7_09,Cy7_08,Cy7_07,Cy7_06,Cy7_05,Cy7_04,Cy7_02,Cy7_03,Cy5_09,Cy5_08,Cy5_07,Cy5_06,Cy5_05,Cy5_04,Cy5_03,Cy5_02
0,Igf1,0,0,1,0,0,0,0,0,0,1,1,0,0,0,1,0
1,Cps1,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0
2,Glul,0,0,0,0,1,1,0,1,0,0,0,1,0,0,0,0
3,Glud1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,1,0
4,Pigr,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,Blank3,1,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0
136,Blank4,0,1,0,0,1,0,0,0,0,0,0,1,1,0,0,0
137,Blank5,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1
138,Blank6,1,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0


### Load Polaris multiplex spot detection model

In [7]:
multiplex_app = Polaris(image_type='multiplex', 
                        #segmentation_type='mesmer',
                        decoding_kwargs={'rounds': rounds, 'channels': channels,
                                        'df_barcodes': df_barcodes})

INFO:root:Checking for cached data
INFO:root:Checking SpotDetection-8.tar.gz against provided file_hash...
INFO:root:SpotDetection-8.tar.gz with hash a6164e48ef8872a9524b4ec6726859d7 already available.
INFO:root:Extracting <user_home>\.deepcell\models\SpotDetection-8.tar.gz
INFO:root:Successfully extracted <user_home>\.deepcell\models\SpotDetection-8.tar.gz into <user_home>\.deepcell\models


In [8]:
print(multiplex_app.distribution)

Relaxed Bernoulli


Load related functions below

In [10]:
def _upsampled_dft(data, upsampled_region_size,
                   upsample_factor=1, axis_offsets=None):
    """
    Upsampled DFT by matrix multiplication.
    This code is intended to provide the same result as if the following
    operations were performed:
        - Embed the array "data" in an array that is ``upsample_factor`` times
          larger in each dimension.  ifftshift to bring the center of the
          image to (1,1).
        - Take the FFT of the larger array.
        - Extract an ``[upsampled_region_size]`` region of the result, starting
          with the ``[axis_offsets+1]`` element.
    It achieves this result by computing the DFT in the output array without
    the need to zeropad. Much faster and memory efficient than the zero-padded
    FFT approach if ``upsampled_region_size`` is much smaller than
    ``data.size * upsample_factor``.
    Parameters
    ----------
    data : array
        The input data array (DFT of original data) to upsample.
    upsampled_region_size : integer or tuple of integers, optional
        The size of the region to be sampled.  If one integer is provided, it
        is duplicated up to the dimensionality of ``data``.
    upsample_factor : integer, optional
        The upsampling factor.  Defaults to 1.
    axis_offsets : tuple of integers, optional
        The offsets of the region to be sampled.  Defaults to None (uses
        image center)
    Returns
    -------
    output : ndarray
            The upsampled DFT of the specified region.
    """
    # if people pass in an integer, expand it to a list of equal-sized sections
    if not hasattr(upsampled_region_size, "__iter__"):
        upsampled_region_size = [upsampled_region_size, ] * data.ndim
    else:
        if len(upsampled_region_size) != data.ndim:
            raise ValueError("shape of upsampled region sizes must be equal "
                             "to input data's number of dimensions.")

    if axis_offsets is None:
        axis_offsets = [0, ] * data.ndim
    else:
        if len(axis_offsets) != data.ndim:
            raise ValueError("number of axis offsets must be equal to input "
                             "data's number of dimensions.")

    im2pi = 1j * 2 * np.pi

    dim_properties = list(zip(data.shape, upsampled_region_size, axis_offsets))

    for (n_items, ups_size, ax_offset) in dim_properties[::-1]:
        kernel = ((np.arange(ups_size) - ax_offset)[:, None]
                  * np.fft.fftfreq(n_items, upsample_factor))
        kernel = np.exp(-im2pi * kernel)

        # Equivalent to:
        #   data[i, j, k] = kernel[i, :] @ data[j, k].T
        data = np.tensordot(kernel, data, axes=(1, -1))
    return data



def register_translation(src_image: np.ndarray,
                         target_image: np.ndarray,
                         upsample_factor: int = 1,
                         space: str = "real",
                         return_error: bool = True,
                         ):
    """
    Efficient subpixel image translation registration by cross-correlation.
    This code gives the same precision as the FFT upsampled cross-correlation
    in a fraction of the computation time and with reduced memory requirements.
    It obtains an initial estimate of the cross-correlation peak by an FFT and
    then refines the shift estimation by upsampling the DFT only in a small
    neighborhood of that estimate by means of a matrix-multiply DFT.
    Parameters
    ----------
    src_image : array
        Reference image.
    target_image : array
        Image to register.  Must be same dimensionality as ``src_image``.
    upsample_factor : int, optional
        Upsampling factor. Images will be registered to within
        ``1 / upsample_factor`` of a pixel. For example
        ``upsample_factor == 20`` means the images will be registered
        within 1/20th of a pixel.  Default is 1 (no upsampling)
    space : string, one of "real" or "fourier", optional
        Defines how the algorithm interprets input data.  "real" means data
        will be FFT'd to compute the correlation, while "fourier" data will
        bypass FFT of input data.  Case insensitive.
    return_error : bool, optional
        Returns error and phase difference if on,
        otherwise only shifts are returned
    Returns
    -------
    shifts : ndarray
        Shift vector (in pixels) required to register ``target_image`` with
        ``src_image``.  Axis ordering is consistent with numpy (e.g. Z, Y, X)
    error : float
        Translation invariant normalized RMS error between ``src_image`` and
        ``target_image``.
    phasediff : float
        Global phase difference between the two images (should be
        zero if images are non-negative).
    References
    ----------
    .. [1] Manuel Guizar-Sicairos, Samuel T. Thurman, and James R. Fienup,
           "Efficient subpixel image registration algorithms,"
           Optics Letters 33, 156-158 (2008). :DOI:`10.1364/OL.33.000156`
    .. [2] James R. Fienup, "Invariant error metrics for image reconstruction"
           Optics Letters 36, 8352-8357 (1997). :DOI:`10.1364/AO.36.008352`
    """
    # images must be the same shape
    if src_image.shape != target_image.shape:
        raise ValueError("Error: images must be same size for "
                         "register_translation")

    # assume complex data is already in Fourier space
    if space.lower() == 'fourier':
        src_freq = src_image
        target_freq = target_image
    # real data needs to be fft'd.
    elif space.lower() == 'real':
        src_freq = np.fft.fftn(src_image)
        target_freq = np.fft.fftn(target_image)
    else:
        raise ValueError("Error: register_translation only knows the \"real\" "
                         "and \"fourier\" values for the ``space`` argument.")

    # Whole-pixel shift - Compute cross-correlation by an IFFT
    shape = src_freq.shape
    image_product = src_freq * target_freq.conj()
    cross_correlation = np.fft.ifftn(image_product)

    # Locate maximum
    maxima = np.unravel_index(np.argmax(np.abs(cross_correlation)),
                              cross_correlation.shape)
    midpoints = np.array([np.fix(axis_size / 2) for axis_size in shape])

    shifts = np.array(maxima, dtype=np.float64)
    shifts[shifts > midpoints] -= np.array(shape)[shifts > midpoints]

    # calculate the pixel-resolution maximum cross-correlation value
    CCmax_pixel = cross_correlation[maxima]

    if upsample_factor == 1:
        if return_error:
            src_amp = np.sum(np.abs(src_freq) ** 2) / src_freq.size
            target_amp = np.sum(np.abs(target_freq) ** 2) / target_freq.size
            CCmax = CCmax_pixel # this is height of cc peak

    # If upsampling > 1, then refine estimate with matrix multiply DFT
    else:
        # Initial shift estimate in upsampled grid
        shifts = np.round(shifts * upsample_factor) / upsample_factor
        upsampled_region_size = np.ceil(upsample_factor * 1.5)

        # Center of output array at dftshift + 1
        dftshift = np.fix(upsampled_region_size / 2.0)
        upsample_factor = np.array(upsample_factor, dtype=np.float64)
        normalization = (src_freq.size * upsample_factor ** 2)

        # Matrix multiply DFT around the current shift estimate
        sample_region_offset = dftshift - shifts * upsample_factor
        cross_correlation = _upsampled_dft(image_product.conj(),
                                           upsampled_region_size,
                                           upsample_factor,
                                           sample_region_offset).conj()
        cross_correlation /= normalization

        # Locate maximum and map back to original pixel grid
        maxima = np.unravel_index(np.argmax(np.abs(cross_correlation)),
                                  cross_correlation.shape)
        CCmax = cross_correlation[maxima]

        maxima = np.array(maxima, dtype=np.float64) - dftshift

        shifts = shifts + maxima / upsample_factor

        if return_error:
            src_amp = _upsampled_dft(src_freq * src_freq.conj(),
                                     1, upsample_factor)[0, 0]
            src_amp /= normalization
            target_amp = _upsampled_dft(target_freq * target_freq.conj(),
                                        1, upsample_factor)[0, 0]
            target_amp /= normalization

    # For singleton dimensions, the shift calculated has no effect.
    # Set to zero.
    for dim in range(src_freq.ndim):
        if shape[dim] == 1:
            shifts[dim] = 0

    if return_error:
        # return (shifts,
        #         _compute_error(CCmax, src_amp, target_amp),
        #         _compute_phasediff(CCmax)
        #         )

        return (shifts,
                np.abs(CCmax),
                np.abs(CCmax_pixel)
                )
    else:
        return shifts
def parseDaybook(filepath: str,
                 verbose: bool = True,
                 ) -> AffineTransform:
    """
    parse daybook lateral chromatic shift output file
    and generate affine matrix

    returns affine matrix ready for warping with skimage
    """
    title_row = None
    values_row = None

    with open(filepath) as csv_file:

        csv_reader = csv.reader(csv_file, delimiter=';')

        for row in csv_reader:
            if "a_affine_parameter" in row:
                title_row = [title.strip() for title in row]
            # go to next row. then stop reading
            elif title_row is not None:
                values_row = [float(value) for value in row]
                break

    assert title_row is not None, "title row for param values not found in daybook csv file"
    assert values_row is not None, "row containing param values not found in daybook csv file"

    matrix = np.zeros((3, 3), dtype=np.float64)
    matrix[0, 0] = values_row[title_row.index("a_affine_parameter")]
    matrix[0, 1] = values_row[title_row.index("b_affine_parameter")]
    matrix[1, 0] = values_row[title_row.index("c_affine_parameter")]
    matrix[1, 1] = values_row[title_row.index("d_affine_parameter")]
    matrix[0, 2] = values_row[title_row.index("x_translation_affine_parameter")]
    matrix[1, 2] = values_row[title_row.index("y_translation_affine_parameter")]
    matrix[2, 2] = 1

    if verbose:
        print("____ Matrix: ____\n", matrix)

    return AffineTransform(matrix)

In [ ]:
import copy
from skimage.transform import warp as skimage_warp
import skimage
from scipy import ndimage

reference_bit = 0 # bit for image registration/alignment
upsampling = 50
# normalization = None
freq_filter = None
verbose = True
type_to_colour = {
    "Cy3": "558",
    "Cy5": "684",
    "Cy7": "776"
}
reference_colour = "684"
calibration_path = os.path.join(main_dir, 'Data/100uM/calibration')
color_list = ['Cy7']*8 +['Cy5']*8
dropped_bits = []
ref_slice_fourier = None

params_pattern1 = re.compile("[a-zA-Z]+(\d+)_[a-zA-Z]+(\d+)")
params_pattern2 = re.compile("([a-zA-Z0-9]+)to([a-zA-Z0-9]+)")

pairs_tf_dict = {}

output_path = os.path.join(main_dir, 'Data/benchmarking/polaris/cell/')
save_regi_output = os.path.join(output_path, f'shift_registration_polaris_GS.txt')
os.makedirs(os.path.dirname(save_regi_output), exist_ok=True)

for file in os.listdir(calibration_path):

    if os.path.splitext(file)[-1] == ".csv":

        match1 = re.search(params_pattern1, file)
        match2 = re.search(params_pattern2, file)

        if match1:
            match = match1
        elif match2:
            match = match2
        else:
            continue

        daybook_fullpath = os.path.join(calibration_path, file)
        colour_pair = (match.group(1), match.group(2))

        pairs_tf_dict[colour_pair] = parseDaybook(daybook_fullpath)
colours_tf_dict = {reference_colour: None}
for pair in pairs_tf_dict:
    if pair[0] == reference_colour:
        colours_tf_dict[pair[1]] = pairs_tf_dict[pair]
    if pair[1] == reference_colour:
        colours_tf_dict[pair[0]] = pairs_tf_dict[pair].inverse



## loop through each FOV
for fov in range(0,25):
    print('processing fov: ' + str(fov))
    dax_image = np.zeros((2048,2048,16)) # for registration
    final_dax_img = np.zeros((1,2048,2048,16))
    # Load 16 bits array into dax_regi dictionary, key correspond to respective bits for registration
    #dax_regi = {} #np.zeros((1,2048,2048,16))
    for i, info in enumerate(channel_hyb): #because there are 12 hybs 2 channels 
        dax_name = str(info) + '_' + f"{fov:02d}" + '.dax'
        print('Processing ' + str(data_path + dax_name))
        dax_image[...,i] = readDoryImg(data_path + dax_name)
        print(f'image shape {dax_image[...,i].shape}')
        #dax_regi[i] = np.expand_dims(image_data, axis=[0, -1]) #from 2048*2048 to 1*2048*2048*1

    ### insert Guidestar chromatic distortion correction ###
    for bit, colour in enumerate(color_list):
        colour = color_list[bit]
        print(
            f"Correcting chromatic distortion for bit {bit} with colour {colour}...\n"
        )

        if bit not in dropped_bits:
            transform = colours_tf_dict[type_to_colour[colour]]
            if transform is not None:
                slice_to_warp = dax_image[...,bit]
                align_slice = skimage_warp(slice_to_warp, transform)
                align_slice = np.expand_dims(align_slice, axis=0)
                dax_image[..., bit] = align_slice
    

    ### insert Guidestar registration ###
    ### modified to eliminate imagestack data structure ###
    #prepare reference img
    ref_slice = copy.copy(dax_image[..., reference_bit])
    ref_slice_fourier = np.fft.fftn(ref_slice)

    for bit in range(len(channel_hyb)):
        target_image = dax_image[..., bit]
        if len(target_image.shape) != 2:
            raise ValueError(
                f"Not 2D image! "
            )
        current_slice = copy.copy(target_image)
        current_slice_fourier = np.fft.fftn(current_slice)


        ### prepare reference bit
        if bit == reference_bit:
            if freq_filter is not None:
                ref_filtered_slice = np.fft.ifftn(ref_slice_fourier * filter_shifted) #fourier filter_shift 
                final_dax_img[..., reference_bit] = ref_filtered_slice.real #save the filtered shift result
            else:
                final_dax_img[..., reference_bit] = dax_image[..., reference_bit]
            continue

        (shifts,
        fine_error,
        pixel_error) = register_translation(ref_slice_fourier,
                                            current_slice_fourier,
                                            upsample_factor=upsampling,
                                            space="fourier")
        with open(save_regi_output, 'a') as f:
            formatted_shifts = ', '.join([f'{x:.3f}' for x in shifts])
            f.write(f"Bit {bit} in fov: {fov} shift merfish by {formatted_shifts}\n")
        ## align image after registration
        registered_slice = np.fft.ifftn(
                        ndimage.fourier_shift(current_slice_fourier, shifts)
                    )
        final_img = registered_slice.real
        
        if freq_filter is not None:
                registered_filtered_slice = np.fft.ifftn(
                    ndimage.fourier_shift(current_slice_fourier * filter_shifted,
                                                shifts)
                        )
                final_img = registered_filtered_slice.real
        final_dax_img[..., bit] = final_img

    save_path = os.path.join(main_dir, 'Data/benchmarking/polaris/cell/') + str(multiplex_app.distribution) + '_thresh0.85_GS_chrco_img_regi_final'
    os.makedirs(save_path, exist_ok=True)
    # np.save(save_path + '/' + str(fov) + save_filename + '.npy', array_to_save)

    background_image = np.max(final_dax_img, axis=-1, keepdims=True)
    mask = mask_spots(background_image, 0.99)

    multiplex_pred = multiplex_app.predict(spots_image=final_dax_img,
                                            #segmentation_image=mesmer_image,
                                            mask=mask,
                                            image_mpp=0.11, clip=True, 
                                            threshold=0.85)
    df_results = multiplex_pred[0]
    segmentation_result = multiplex_pred[1]
    df_results.to_csv(save_path + '/' + str(fov) + '_Polaris_prediction.csv')

____ Matrix: ____
 [[ 1.00130105e+00  4.67026129e-06 -1.43742820e+00]
 [-4.67026129e-06  1.00130105e+00 -1.22251447e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
____ Matrix: ____
 [[ 1.00177926e+00  2.50579395e-05 -1.69703024e+00]
 [-2.50579395e-05  1.00177926e+00 -1.93060367e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
____ Matrix: ____
 [[ 9.97184823e-01 -2.92860502e-05  2.96042144e+00]
 [ 2.92860502e-05  9.97184823e-01  2.68954620e+00]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
processing fov: 0
Processing <main_dir>/Data/100uM/Cy7_09_00.dax
image shape (2048, 2048)
Processing <main_dir>/Data/100uM/Cy7_08_00.dax
image shape (2048, 2048)
Processing <main_dir>/Data/100uM/Cy7_07_00.dax
image shape (2048, 2048)
Processing <main_dir>/Data/100uM/Cy7_06_00.dax
image shape (2048, 2048)
Processing <main_dir>/Data/100uM/Cy7_05_00.dax
image shape (2048, 2048)
Processing <main_dir>/Data/100uM/Cy7_04_00.dax
image shape (2048, 2048)
Processing <main_dir>/Data/100u

100%|██████████| 16/16 [10:20<00:00, 38.81s/it]
<user_home>\miniforge3\envs\Polaris38\lib\site-packages\torch\__init__.py:955: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\tensor\python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)


Decoding gene identities.
Training...


100%|██████████| 500/500 [00:28<00:00, 17.26it/s]


Estimating barcode probabilities...
Revising errors...


481368it [16:14, 494.00it/s]


... (141 outputs left out)

100%|██████████| 500/500 [00:31<00:00, 15.96it/s]


Estimating barcode probabilities...
Revising errors...


885927it [28:24, 519.76it/s] 


390947 of 649046 rescue attempts were successful.
Refining spot locations.
